In [29]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Bidirectional, GRU, Dense, Dropout, Layer
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
import tensorflow.keras.backend as K
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report
import matplotlib.pyplot as plt
import tensorflow as tf


In [30]:
import pandas as pd

df = pd.read_csv('dataset_LOSO/train.csv')
df.head()

,label,NOSE_x,NOSE_y,NOSE_z,NOSE_visibility,LEFT_SHOULDER_x,LEFT_SHOULDER_y,LEFT_SHOULDER_z,LEFT_SHOULDER_visibility,RIGHT_SHOULDER_x,...,RIGHT_KNEE_visibility,LEFT_ANKLE_x,LEFT_ANKLE_y,LEFT_ANKLE_z,LEFT_ANKLE_visibility,RIGHT_ANKLE_x,RIGHT_ANKLE_y,RIGHT_ANKLE_z,RIGHT_ANKLE_visibility,SUB_ID
0,1,0.492359,0.288014,-0.252744,0.999994,0.535406,0.374350,-0.077348,0.999983,0.452382,...,0.994587,0.512328,0.912481,0.160206,0.988206,0.486275,0.914611,0.169390,0.993024,0
1,1,0.492433,0.289817,-0.229027,0.999993,0.535407,0.374384,-0.048240,0.999980,0.452379,...,0.994670,0.512457,0.913148,0.160463,0.988055,0.486412,0.915192,0.144432,0.992985,0
2,1,0.492499,0.293093,-0.235932,0.999992,0.535386,0.375816,-0.055595,0.999979,0.452351,...,0.994697,0.512681,0.913221,0.166236,0.987879,0.486538,0.915200,0.150508,0.992782,0
3,1,0.492648,0.296606,-0.253789,0.999988,0.535401,0.381458,-0.057004,0.999971,0.451909,...,0.994679,0.512805,0.913556,0.176160,0.987470,0.486680,0.915507,0.166394,0.992556,0
4,1,0.492675,0.302113,-0.244568,0.999983,0.535405,0.387984,-0.055096,0.999960,0.451867,...,0.994668,0.512987,0.914756,0.179059,0.986808,0.486683,0.915941,0.171940,0.992234,0


In [31]:
# Thay 'V' thành 9 trong cột SUB_ID
df['SUB_ID'] = df['SUB_ID'].replace('V', 9)

# ==== 3. Các subject ====
df['SUB_ID'] = df['SUB_ID'].astype(int)
subjects = df['SUB_ID'].unique()
num_classes = len(df['label'].unique())

print(f"Number of subjects: {subjects}")
print(f"Number of classes: {num_classes}")

Number of subjects: [0 1 2 3 4 5 6 7 8 9]
Number of classes: 6


In [32]:
# Định nghĩa lớp Attention
@tf.keras.utils.register_keras_serializable(package='Custom', name='Attention')
class Attention(Layer):
    def __init__(self, **kwargs):
        super(Attention, self).__init__(**kwargs)

    def build(self, input_shape):
        self.W = self.add_weight(name='attention_weight', shape=(input_shape[-1], 1), initializer='random_normal', trainable=True)
        self.b = self.add_weight(name='attention_bias', shape=(input_shape[1], 1), initializer='zeros', trainable=True)
        super(Attention, self).build(input_shape)

    def call(self, x):
        e = K.tanh(K.dot(x, self.W) + self.b)
        a = K.softmax(e, axis=1)
        output = x * a
        return K.sum(output, axis=1)


In [33]:
# ==== 4. Hàm build mô hình Bi-GRU + Attention ====
def build_bi_gru_attention_model(input_shape, num_classes):
    input_layer = Input(shape=input_shape)

    gru_layer = Bidirectional(GRU(64, return_sequences=True, kernel_regularizer='l2'))(input_layer)
    attention_layer = Attention()(gru_layer)  # (query, value)

    dropout_layer = Dropout(0.5)(attention_layer)
    output_layer = Dense(num_classes, activation='softmax')(dropout_layer)

    model = Model(inputs=input_layer, outputs=output_layer)
    optimizer = Adam(learning_rate=0.001)
    model.compile(optimizer=optimizer, 
              loss='categorical_crossentropy', 
              metrics=['accuracy'])
    model.summary()
    return model

In [36]:
import os

# ==== 5. LOSO loop ====
all_reports = []

encoder = OneHotEncoder(sparse_output=False)

for test_subject in subjects:
    print(f'\n===== Testing on subject: {test_subject} =====')

    # Tách dữ liệu
    test_df = df[df['SUB_ID'] == test_subject]
    train_df = df[df['SUB_ID'] != test_subject]

    X_train = train_df.drop(['SUB_ID', 'label'], axis=1).values
    y_train = train_df['label'].values

    X_test = test_df.drop(['SUB_ID', 'label'], axis=1).values
    y_test = test_df['label'].values

    # Chuẩn hóa
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    # Định hình lại cho GRU
    X_train = X_train.reshape((X_train.shape[0], 1, X_train.shape[1]))
    X_test = X_test.reshape((X_test.shape[0], 1, X_test.shape[1]))

    # One-hot encoding cho nhãn
    y_train = encoder.fit_transform(y_train.reshape(-1, 1))
    y_test = encoder.transform(y_test.reshape(-1, 1))

    # Tính class weights (dựa trên nhãn gốc trước khi one-hot)
    y_train_labels = np.argmax(y_train, axis=1)
    class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(y_train_labels), y=y_train_labels)
    class_weights_dict = dict(zip(np.unique(y_train_labels), class_weights))

    # Build model
    model = build_bi_gru_attention_model(input_shape=(X_train.shape[1], X_train.shape[2]), num_classes=num_classes)

    # Callbacks
    os.makedirs("Model", exist_ok=True)
    checkpoint = ModelCheckpoint("Model/Squat_detection_GRU_LOSO.keras", save_best_only=True, monitor="val_loss", mode="min", verbose=0)
    early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=0)

    # Train
    model.fit(
        X_train, y_train,
        validation_split=0.1,
        epochs=50,
        batch_size=32,
        verbose=0,
        class_weight=class_weights_dict,
        callbacks=[checkpoint, early_stopping]
    )

    # Dự đoán và đánh giá
    y_pred = model.predict(X_test)
    y_pred_classes = np.argmax(y_pred, axis=1)
    y_true_classes = np.argmax(y_test, axis=1)

    report = classification_report(y_true_classes, y_pred_classes, output_dict=True)
    all_reports.append(report)


===== Testing on subject: 0 =====


Model: "functional_23"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_24 (InputLayer)     │ (None, 1, 36)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_24                │ (None, 1, 128)         │        39,168 │
│ (Bidirectional)                 │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ attention_24 (Attention)        │ (None, 128)            │           129 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_23 (Dropout)            │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_23 (Dense)                │ (None, 6)              │           774 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 40,071 (156.53 KB)

 Trainable params: 40,071 (156.53 KB)

...

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_33 (InputLayer)     │ (None, 1, 36)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_33                │ (None, 1, 128)         │        39,168 │
│ (Bidirectional)                 │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ attention_33 (Attention)        │ (None, 128)            │           129 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_32 (Dropout)            │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_32 (Dense)                │ (None, 6)              │           774 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 40,071 (156.53 KB)

 Trainable params: 40,071 (156.53 KB)

 Non-trainable params: 0 (0.00 B)

86/86 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step


In [37]:
# Gộp và tính trung bình theo từng nhãn
average_report = pd.concat([pd.DataFrame(r).T for r in all_reports]).groupby(level=0).mean()

print("\n=== Average LOSO Classification Report ===")
print(average_report)


=== Average LOSO Classification Report ===
              precision    recall  f1-score      support
0              0.933808  0.866666  0.896218   754.200000
1              0.978351  0.937471  0.954984   589.000000
2              0.965953  0.973624  0.967893   593.800000
3              0.748426  0.864437  0.784944   289.800000
4              0.789588  0.815008  0.793090   227.900000
5              0.833380  0.776353  0.763556   202.400000
accuracy       0.909041  0.909041  0.909041     0.909041
macro avg      0.874918  0.872260  0.860114  2657.100000
weighted avg   0.929376  0.909041  0.911678  2657.100000
